### 语义检索

In [1]:
import os
import dashscope
from dotenv import load_dotenv

# 加载 .env 文件中的 DASHSCOPE_API_KEY
load_dotenv()
dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")

print("API Key 已设置：", dashscope.api_key[:6] + "********")

API Key 已设置： sk-46a********


In [2]:
from typing import List
from langchain_core.embeddings import Embeddings

class QwenEmbeddings(Embeddings):
    def __init__(self, model_name: str = "qwen3.7-text-embedding"):
        self.model_name = model_name

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """把多个文本转成向量列表"""
        resp = dashscope.TextEmbedding.call(
            model=self.model_name,
            input=texts
        )
        if resp.status_code == 200:
            embeddings = resp.output['embeddings']
            # 按原始顺序提取向量数据
            return [item['embedding'] for item in sorted(embeddings, key=lambda x: x['text_index'])]
        else:
            raise Exception(f"DashScope 接口报错: {resp.code} - {resp.message}")

    def embed_query(self, text: str) -> List[float]:
        """把单条搜索文本转成向量"""
        return self.embed_documents([text])[0]

# --- 验证 Embedding 是否有效 ---
embedding_model = QwenEmbeddings()
test_vec = embedding_model.embed_query("测试文本")

print("Embedding 初始化成功！")
print("测试向量维度 (dims)：", len(test_vec))
print("向量前 3 个数值：", test_vec[:3])

Embedding 初始化成功！
测试向量维度 (dims)： 1024
向量前 3 个数值： [0.029063522815704346, 0.024911589920520782, 0.01725948415696621]


In [3]:
from langgraph.store.memory import InMemoryStore

# 配置存储库
store = InMemoryStore(
    index={
        "embed": embedding_model, # 使用我们在 Cell 2 里定义的模型
        "dims": 1024,             # 向量维度
        "fields": ["text"],       # 告诉 Store：以后传进来的字典里，只有 "text" 字段需要做向量计算
    }
)

print("InMemoryStore 初始化成功！")

InMemoryStore 初始化成功！


In [4]:
# 定义存储的命名空间（区分不同的用户或会话）
namespace = ("users", "user_123", "memories")

# 写入 4 条不同的记忆数据
store.put(
    namespace=namespace,
    key="mem_1",
    value={"text": "阿里云 DashScope 提供高性能的大模型 API 服务。", "category": "cloud"}
)

store.put(
    namespace=namespace,
    key="mem_2",
    value={"text": "苹果是一种富含维生素和膳食纤维的常见水果。", "category": "fruit"}
)

store.put(
    namespace=namespace,
    key="mem_3",
    value={"text": "通义千问是阿里云自主研发的超大规模语言模型。", "category": "ai"}
)

store.put(
    namespace=namespace,
    key="mem_4",
    value={"text": "向量数据库主要用于存储和快速检索高维向量数据。", "category": "database"}
)

print("4 条数据已成功存入 Store 并完成了后台向量化！")

4 条数据已成功存入 Store 并完成了后台向量化！


In [5]:
# 直接通过 key 拿指定的记忆
item_1 = store.get(namespace, "mem_1")

print("Key 为 mem_1 的完整对象：", item_1)
print("提取 value 内容：", item_1.value)

Key 为 mem_1 的完整对象： Item(namespace=['users', 'user_123', 'memories'], key='mem_1', value={'text': '阿里云 DashScope 提供高性能的大模型 API 服务。', 'category': 'cloud'}, created_at='2026-10-04T03:25:56.746817+00:00', updated_at='2026-10-04T03:25:56.746822+00:00')
提取 value 内容： {'text': '阿里云 DashScope 提供高性能的大模型 API 服务。', 'category': 'cloud'}


In [ ]:
query_text = "阿里有哪些人工智能产品？"

# 执行语义检索
results = store.search(
    namespace,          # 指定搜索哪个空间
    query=query_text,   # 搜索的自然语言文本
    limit=2             # 返回相似度最高的前 2 个结果
)

print(f"搜索问题: '{query_text}'\n")
print(f"检索到 {len(results)} 条最相关的记忆：\n" + "="*40)

for i, item in enumerate(results, 1):
    print(f"结果 {i}:")
    print(f"  └─ 记忆 Key  : {item.key}")
    print(f"  └─ 文本内容  : {item.value['text']}")
    print(f"  └─ 匹配得分  : {item.score:.4f}")  # 得分越接近 1.0 说明越相似
    print("-" * 40)